In [2]:
!pip install gensim nltk numpy scikit-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 47.7 MB/s eta 0:00:00:00:010:01


# Word2Vec Tokenization Demonstration

This notebook demonstrates how Word2Vec tokenizes text and learns word embeddings. We'll explore:
- Different tokenization approaches
- Text preprocessing techniques
- Word2Vec vocabulary building
- Word embeddings and semantic relationships

**Key Concepts:**
- **Tokenization**: Breaking text into individual words/tokens
- **Vocabulary**: Collection of unique tokens from the corpus
- **Word Embeddings**: Dense vector representations of words
- **Context Window**: Number of surrounding words used for learning relationships

In [3]:
import re
import string
from collections import Counter
from gensim.models import Word2Vec
from nltk.tokenize import word_tokenize
import nltk

# Download required NLTK data
try:
    nltk.data.find('tokenizers/punkt')
except LookupError:
    nltk.download('punkt')

print("✓ All imports successful!")

[nltk_data] Downloading package punkt to /root/nltk_data...


✓ All imports successful!


[nltk_data]   Unzipping tokenizers/punkt.zip.


## 1. Different Tokenization Approaches

Tokenization is the first step in processing text. Let's explore different methods:

In [6]:
# Download NLTK data explicitly
import nltk
print("Downloading NLTK tokenizer data...")
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)
print("✓ NLTK data downloaded successfully")

✓ NLTK data downloaded successfully


In [7]:
def basic_tokenization(text: str) -> list:
    """Simple tokenization by splitting on whitespace."""
    return text.split()


def advanced_tokenization(text: str) -> list:
    """Tokenization with preprocessing: lowercase, remove punctuation, split."""
    text = text.lower()
    text = text.translate(str.maketrans('', '', string.punctuation))
    tokens = text.split()
    return tokens


def regex_tokenization(text: str) -> list:
    """Tokenization using regex patterns."""
    text = text.lower()
    tokens = re.findall(r"\b\w+(?:'\w+)?\b", text)
    return tokens


def nltk_tokenization(text: str) -> list:
    """Tokenization using NLTK's word_tokenize."""
    text = text.lower()
    tokens = word_tokenize(text)
    tokens = [t for t in tokens if t.isalnum()]
    return tokens

print("✓ Tokenization functions defined")

✓ Tokenization functions defined


In [8]:
# Sample text for demonstration
sample_text = (
    "Natural language processing is fascinating! "
    "Word2Vec models understand word relationships. "
    "Tokenization is the first step in NLP."
)

print("ORIGINAL TEXT:")
print(f"  {sample_text}\n")

# Compare different tokenization methods
print("=" * 70)
print("TOKENIZATION COMPARISON")
print("=" * 70)

# Basic tokenization
basic_tokens = basic_tokenization(sample_text)
print(f"\n1. BASIC TOKENIZATION (whitespace split):")
print(f"   Tokens: {basic_tokens}")
print(f"   Count: {len(basic_tokens)}")

# Advanced tokenization
advanced_tokens = advanced_tokenization(sample_text)
print(f"\n2. ADVANCED TOKENIZATION (lowercase + remove punctuation):")
print(f"   Tokens: {advanced_tokens}")
print(f"   Count: {len(advanced_tokens)}")

# Regex tokenization
regex_tokens = regex_tokenization(sample_text)
print(f"\n3. REGEX TOKENIZATION (word patterns):")
print(f"   Tokens: {regex_tokens}")
print(f"   Count: {len(regex_tokens)}")

# NLTK tokenization
nltk_tokens = nltk_tokenization(sample_text)
print(f"\n4. NLTK TOKENIZATION (NLTK word_tokenize):")
print(f"   Tokens: {nltk_tokens}")
print(f"   Count: {len(nltk_tokens)}")

ORIGINAL TEXT:
  Natural language processing is fascinating! Word2Vec models understand word relationships. Tokenization is the first step in NLP.

TOKENIZATION COMPARISON

1. BASIC TOKENIZATION (whitespace split):
   Tokens: ['Natural', 'language', 'processing', 'is', 'fascinating!', 'Word2Vec', 'models', 'understand', 'word', 'relationships.', 'Tokenization', 'is', 'the', 'first', 'step', 'in', 'NLP.']
   Count: 17

2. ADVANCED TOKENIZATION (lowercase + remove punctuation):
   Tokens: ['natural', 'language', 'processing', 'is', 'fascinating', 'word2vec', 'models', 'understand', 'word', 'relationships', 'tokenization', 'is', 'the', 'first', 'step', 'in', 'nlp']
   Count: 17

3. REGEX TOKENIZATION (word patterns):
   Tokens: ['natural', 'language', 'processing', 'is', 'fascinating', 'word2vec', 'models', 'understand', 'word', 'relationships', 'tokenization', 'is', 'the', 'first', 'step', 'in', 'nlp']
   Count: 17

4. NLTK TOKENIZATION (NLTK word_tokenize):
   Tokens: ['natural', 'langu

## 2. Vocabulary Building & Frequency Analysis

After tokenization, we build a vocabulary of unique tokens and analyze their frequencies:

In [9]:
# Build vocabulary from advanced tokens
token_freq = Counter(advanced_tokens)

print("VOCABULARY ANALYSIS:")
print(f"Total unique tokens: {len(token_freq)}")
print(f"Total tokens: {len(advanced_tokens)}\n")

print("Token Frequencies:")
print("-" * 40)
for token, freq in sorted(token_freq.items(), key=lambda x: x[1], reverse=True):
    bar = "█" * freq
    print(f"  {token:15} : {bar} ({freq})")

VOCABULARY ANALYSIS:
Total unique tokens: 16
Total tokens: 17

Token Frequencies:
----------------------------------------
  is              : ██ (2)
  natural         : █ (1)
  language        : █ (1)
  processing      : █ (1)
  fascinating     : █ (1)
  word2vec        : █ (1)
  models          : █ (1)
  understand      : █ (1)
  word            : █ (1)
  relationships   : █ (1)
  tokenization    : █ (1)
  the             : █ (1)
  first           : █ (1)
  step            : █ (1)
  in              : █ (1)
  nlp             : █ (1)


## 3. Word2Vec Preprocessing

For Word2Vec training, we need to preprocess text into sentences of tokens:

In [10]:
def preprocess_sentences(sentences: list) -> list:
    """Preprocess sentences for word2vec training."""
    processed = []
    for sentence in sentences:
        # Lowercase
        sentence = sentence.lower()
        # Remove punctuation
        sentence = re.sub(r'[^\w\s]', '', sentence)
        # Tokenize
        tokens = sentence.split()
        # Filter: remove short tokens
        tokens = [t for t in tokens if len(t) > 2]
        processed.append(tokens)
    return processed


# Sample sentences for Word2Vec
sentences = [
    "Natural language processing is fascinating.",
    "Word2Vec models understand word relationships.",
    "Tokenization is the first step in NLP.",
    "Word embeddings capture semantic meaning.",
    "Machine learning uses word vectors.",
    "Deep learning processes text data.",
    "Context windows help models learn relationships.",
    "Skip-gram predicts context from target words.",
]

print("PREPROCESSING FOR WORD2VEC:")
print(f"Original sentences: {len(sentences)}\n")

processed_sentences = preprocess_sentences(sentences)

print("Preprocessed sentences:")
print("-" * 70)
for i, (orig, proc) in enumerate(zip(sentences, processed_sentences), 1):
    print(f"  {i}. Original: {orig}")
    print(f"     Tokens:   {proc}\n")

PREPROCESSING FOR WORD2VEC:
Original sentences: 8

Preprocessed sentences:
----------------------------------------------------------------------
  1. Original: Natural language processing is fascinating.
     Tokens:   ['natural', 'language', 'processing', 'fascinating']

  2. Original: Word2Vec models understand word relationships.
     Tokens:   ['word2vec', 'models', 'understand', 'word', 'relationships']

  3. Original: Tokenization is the first step in NLP.
     Tokens:   ['tokenization', 'the', 'first', 'step', 'nlp']

  4. Original: Word embeddings capture semantic meaning.
     Tokens:   ['word', 'embeddings', 'capture', 'semantic', 'meaning']

  5. Original: Machine learning uses word vectors.
     Tokens:   ['machine', 'learning', 'uses', 'word', 'vectors']

  6. Original: Deep learning processes text data.
     Tokens:   ['deep', 'learning', 'processes', 'text', 'data']

  7. Original: Context windows help models learn relationships.
     Tokens:   ['context', 'windows', 'h

## 4. Training Word2Vec Model

Word2Vec learns word embeddings by predicting context words from target words (Skip-gram) or vice versa (CBOW):

**Key Parameters:**
- **vector_size**: Dimensions of word embeddings (e.g., 100)
- **window**: Context window size (number of words left/right)
- **min_count**: Minimum word frequency to include in vocabulary
- **seed**: For reproducibility

In [11]:
# Train Word2Vec model
print("TRAINING WORD2VEC MODEL...")
print("-" * 70)

model = Word2Vec(
    sentences=processed_sentences,
    vector_size=100,      # 100-dimensional embeddings
    window=5,             # 5 words left/right context
    min_count=1,          # Include all words
    workers=4,            # Use 4 threads
    seed=42,              # Reproducible
    sg=1                  # 1=Skip-gram, 0=CBOW
)

print(f"✓ Model trained successfully!")
print(f"  - Vocabulary size: {len(model.wv)}")
print(f"  - Vector dimensions: {model.wv.vector_size}")
print(f"  - Training epochs: {model.epochs}\n")

print("Vocabulary (all unique tokens learned):")
print("-" * 70)
vocab_list = list(model.wv.index_to_key)
for i, word in enumerate(vocab_list, 1):
    print(f"  {i:2d}. '{word}'")

TRAINING WORD2VEC MODEL...
----------------------------------------------------------------------
✓ Model trained successfully!
  - Vocabulary size: 35
  - Vector dimensions: 100
  - Training epochs: 5

Vocabulary (all unique tokens learned):
----------------------------------------------------------------------
   1. 'word'
   2. 'context'
   3. 'learning'
   4. 'relationships'
   5. 'models'
   6. 'words'
   7. 'target'
   8. 'from'
   9. 'predicts'
  10. 'skipgram'
  11. 'learn'
  12. 'help'
  13. 'windows'
  14. 'data'
  15. 'text'
  16. 'processes'
  17. 'deep'
  18. 'vectors'
  19. 'uses'
  20. 'machine'
  21. 'meaning'
  22. 'semantic'
  23. 'capture'
  24. 'embeddings'
  25. 'nlp'
  26. 'step'
  27. 'first'
  28. 'the'
  29. 'tokenization'
  30. 'understand'
  31. 'word2vec'
  32. 'fascinating'
  33. 'processing'
  34. 'language'
  35. 'natural'


## 5. Exploring Word Embeddings

Each word is now represented as a dense vector of learned embeddings:

In [12]:
import numpy as np

# Show word vector for a sample word
sample_word = 'word'

print(f"WORD EMBEDDINGS:")
print(f"Sample word: '{sample_word}'\n")

try:
    vector = model.wv[sample_word]
    print(f"Vector shape: {vector.shape}")
    print(f"Vector type: {type(vector)}\n")
    
    print(f"First 15 dimensions of '{sample_word}' vector:")
    print(f"  {vector[:15]}\n")
    
    print(f"Vector statistics:")
    print(f"  Min value:  {vector.min():.4f}")
    print(f"  Max value:  {vector.max():.4f}")
    print(f"  Mean:       {vector.mean():.4f}")
    print(f"  Std dev:    {vector.std():.4f}")
    
except KeyError:
    print(f"  '{sample_word}' not in vocabulary")

WORD EMBEDDINGS:
Sample word: 'word'

Vector shape: (100,)
Vector type: <class 'numpy.ndarray'>

First 15 dimensions of 'word' vector:
  [-0.00821498  0.00547912  0.00309143 -0.00122243 -0.0013397   0.00717196
 -0.00828109  0.00394736 -0.00597061 -0.00811645  0.00052958  0.00951245
  0.00471505  0.00522279  0.00434955]

Vector statistics:
  Min value:  -0.0091
  Max value:  0.0095
  Mean:       0.0005
  Std dev:    0.0054


## 6. Word Similarity & Semantic Relationships

Word2Vec learns semantic relationships. Similar words have similar vectors:

In [13]:
print("WORD SIMILARITY ANALYSIS:")
print("=" * 70)

# Find similar words for different target words
test_words = ['word', 'learning', 'data']

for test_word in test_words:
    print(f"\nWords most similar to '{test_word}':")
    print("-" * 70)
    
    try:
        similar_words = model.wv.most_similar(test_word, topn=5)
        for rank, (word, similarity) in enumerate(similar_words, 1):
            bar = "█" * int(similarity * 20)
            print(f"  {rank}. '{word:12}' - Similarity: {similarity:.4f} {bar}")
    except KeyError:
        print(f"  '{test_word}' not in vocabulary")

WORD SIMILARITY ANALYSIS:

Words most similar to 'word':
----------------------------------------------------------------------
  1. 'vectors     ' - Similarity: 0.2154 ████
  2. 'predicts    ' - Similarity: 0.1931 ███
  3. 'nlp         ' - Similarity: 0.1439 ██
  4. 'text        ' - Similarity: 0.1406 ██
  5. 'deep        ' - Similarity: 0.1285 ██

Words most similar to 'learning':
----------------------------------------------------------------------
  1. 'predicts    ' - Similarity: 0.1658 ███
  2. 'uses        ' - Similarity: 0.1281 ██
  3. 'relationships' - Similarity: 0.1045 ██
  4. 'data        ' - Similarity: 0.0969 █
  5. 'vectors     ' - Similarity: 0.0801 █

Words most similar to 'data':
----------------------------------------------------------------------
  1. 'target      ' - Similarity: 0.2286 ████
  2. 'language    ' - Similarity: 0.1711 ███
  3. 'deep        ' - Similarity: 0.1411 ██
  4. 'nlp         ' - Similarity: 0.1394 ██
  5. 'the         ' - Similarity: 0.1373 █

## 7. Summary: From Text to Embeddings

**Complete Word2Vec Pipeline:**

```
Raw Text
    ↓
Preprocessing (lowercase, remove punctuation, etc.)
    ↓
Tokenization (split into words)
    ↓
Vocabulary Building (collect unique tokens)
    ↓
Word2Vec Training:
  - Each word becomes a vector
  - Context windows help learn relationships
  - Similar words get similar vectors
    ↓
Learned Embeddings (100-dimensional vectors per word)
    ↓
Downstream Tasks:
  - Similarity calculations
  - Semantic analysis
  - Word clustering
  - Text classification
```

**Key Insights:**

1. **Tokenization is critical**: Different approaches affect vocabulary quality
2. **Preprocessing matters**: Lowercase, punctuation removal improves training
3. **Context windows**: Larger windows capture broader semantic meaning
4. **Embeddings are learned**: Values adjust during training to minimize prediction loss
5. **Semantic relationships emerge**: Similar words naturally cluster in vector space

## References

- **Word2Vec Paper**: Mikolov et al., "Efficient Estimation of Word Representations in Vector Space" (2013)
- **Gensim Documentation**: https://radimrehurek.com/gensim/
- **Skip-gram Model**: Predicts context words from target word
- **CBOW Model**: Predicts target word from context words

## Further Exploration

- Experiment with different `vector_size` values (50, 100, 300)
- Vary the `window` parameter to see how context affects embeddings
- Use larger corpora for better semantic learning
- Visualize embeddings using t-SNE or UMAP
- Try analogies: "king - man + woman ≈ queen"

## 8. Understanding Word2Vec: Static vs. Contextual Vectors

### Key Insight: Word2Vec Produces **Static, Word-Level Vectors**

Unlike modern contextual models (BERT, GPT), Word2Vec assigns **one fixed vector to each word**, regardless of context.

**What does this mean?**
- Every occurrence of a word gets the **same vector**
- The word "bank" has one representation:
  - "Bank of the river" → same vector as "Bank account"
  - This is both a **strength** and a **limitation**

### Static vs. Contextual Models

**Word2Vec (Static):**
```
Word embedding: word → ALWAYS the same vector
Context-aware: ❌ No
Speed: ⚡ Very fast
Model size: 💾 Lightweight (100-300 dims per word)
```

**Contextual Models (BERT, ELMo, GPT):**
```
Word embedding: context-dependent different vectors
Context-aware: ✓ Yes
Speed: Slower
Model size: Much larger (need full neural network)
```

### The Polysemy Problem

**Polysemy** = One word with multiple meanings

Word2Vec cannot distinguish:
- "The bank is by the river" (financial institution)
- "I work at the bank" (riverbank)

Both use the **same vector**, missing the semantic difference!

### Characteristics of Word2Vec

| Aspect | Details |
|--------|---------|
| **Vector Type** | Dense (100-300 dimensions) |
| **Scope** | Word-level (one per word) |
| **Static?** | Yes (fixed after training) |
| **Context-aware** | No (ignores surrounding words) |
| **Handles Polysemy** | No (same vector for all meanings) |
| **Computational Cost** | Low (simple lookup) |
| **Use Case** | Fast similarity, document classification |
| **Limitation** | Can't capture word sense disambiguation |

In [14]:
print("=" * 80)
print("DEMONSTRATING STATIC VECTORS")
print("=" * 80)

# Example: Word "learning" in different contexts
word_test = 'learning'

if word_test in model.wv:
    print(f"\nWord: '{word_test}'\n")
    
    # Get the vector
    vector1 = model.wv[word_test]
    vector2 = model.wv[word_test]
    
    print(f"Vector 1: {vector1[:10]}  (first 10 dimensions)")
    print(f"Vector 2: {vector2[:10]}  (first 10 dimensions)")
    print(f"\nAre they identical? {np.allclose(vector1, vector2)}")
    
    print("\n" + "-" * 80)
    print("CONTEXTS WHERE THIS WORD MIGHT APPEAR:")
    print("-" * 80)
    print("  1. 'Machine learning uses word vectors'")
    print("  2. 'I am learning Python today'")
    print("  3. 'Deep learning processes text data'")
    print("\n→ In ALL contexts, the word 'learning' gets the SAME vector!")
    print("→ Word2Vec cannot distinguish the different meanings")
    
else:
    print(f"'{word_test}' not in vocabulary")

print("\n" + "=" * 80)
print("COMPARISON: CONTEXTUAL vs STATIC")
print("=" * 80)

comparison_text = """
STATIC (Word2Vec):
  Input: "learning" in any sentence
  Output: [0.45, -0.23, 0.87, ...] (always the same)
  
CONTEXTUAL (BERT/GPT):
  Input: "Machine learning" 
  Output: [0.45, -0.23, 0.87, ...]
  
  Input: "I am learning Python"
  Output: [0.52, -0.18, 0.79, ...] (different!)
  
  → Different context, different vectors
"""
print(comparison_text)

DEMONSTRATING STATIC VECTORS

Word: 'learning'

Vector 1: [-0.00266072  0.00817147 -0.00010723  0.00399739 -0.00084427 -0.00468929
  0.00528627  0.00937374 -0.00472208  0.00556696]  (first 10 dimensions)
Vector 2: [-0.00266072  0.00817147 -0.00010723  0.00399739 -0.00084427 -0.00468929
  0.00528627  0.00937374 -0.00472208  0.00556696]  (first 10 dimensions)

Are they identical? True

--------------------------------------------------------------------------------
CONTEXTS WHERE THIS WORD MIGHT APPEAR:
--------------------------------------------------------------------------------
  1. 'Machine learning uses word vectors'
  2. 'I am learning Python today'
  3. 'Deep learning processes text data'

→ In ALL contexts, the word 'learning' gets the SAME vector!
→ Word2Vec cannot distinguish the different meanings

COMPARISON: CONTEXTUAL vs STATIC

STATIC (Word2Vec):
  Input: "learning" in any sentence
  Output: [0.45, -0.23, 0.87, ...] (always the same)
  
CONTEXTUAL (BERT/GPT):
  Input: "M

## 9. When to Use Word2Vec vs. Contextual Models

### Use Word2Vec When:
✓ You need **fast inference** (real-time applications)  
✓ You have **limited computational resources** (phones, edge devices)  
✓ You need **interpretable** word similarities  
✓ You're doing **document classification** or **clustering**  
✓ You want **lightweight models** for deployment  
✓ Your task **doesn't require context understanding**  

### Use Contextual Models (BERT, GPT) When:
✓ You need **semantic understanding** (handling polysemy)  
✓ You're doing **named entity recognition** or **POS tagging**  
✓ You need **question answering** or **machine translation**  
✓ You have **sufficient computational resources**  
✓ Your task requires **context-dependent meanings**  

### Summary: Static Nature of Word2Vec

**Word2Vec is fundamentally limited by its static, word-level design:**

1. **One vector per word** (regardless of context)
2. **Cannot handle word sense disambiguation** (polysemy)
3. **Fast but less accurate** for complex NLP tasks
4. **Perfect for simple similarity tasks** and **document embeddings**

**Modern Alternative:**
- Use contextual models for complex understanding
- Use Word2Vec for efficiency and simplicity
- Hybrid approach: Use Word2Vec embeddings as input to contextual models